<a href="https://colab.research.google.com/github/ankurdev1-drth/Deep_Learning-/blob/main/02_Training_Deep_Neural_Networks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Setup

In [1]:
import torch.nn as nn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
import torch

layer = nn.Linear(40, 10)
layer.weight.data *= 6 ** 0.5  # kaiming init (or 3 ** 0.5 for LeCun)
torch.zero_(layer.bias.data)

tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])

In [3]:
layer.weight.shape

torch.Size([10, 40])

In [4]:
layer.bias.shape

torch.Size([10])

In [5]:
# another way to for initialization
nn.init.kaiming_uniform_(layer.weight)
nn.init.zeros_(layer.bias)

Parameter containing:
tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.], requires_grad=True)

This is clearer and less error prone as compared to the first approach

For applying initialization to the weights of every `nn.Linear` layer in a model the simples option is to write a little function that takes a module, checks whether it's an instance of the `nn.Linear` class, and if so , applies the desired initialization function to its weights. and the function can be then applied using the `apply()` method

In [6]:
def use_he_init(module):
  if isinstance(module, nn.Linear):
    nn.init.kaiming_uniform_(module.weight)
    nn.init.zeros_(module.bias)

module = nn.Sequential(
    nn.Linear(40, 10),
    nn.ReLU(),
    nn.Linear(10, 1),
    nn.ReLU()
    )
module.apply(use_he_init)

Sequential(
  (0): Linear(in_features=40, out_features=10, bias=True)
  (1): ReLU()
  (2): Linear(in_features=10, out_features=1, bias=True)
  (3): ReLU()
)

### Leaky ReLU

In [7]:
alpha = 0.2
model = nn.Sequential(
    nn.Linear(50, 40), #0
    nn.LeakyReLU(negative_slope=alpha), #1
    nn.Linear(40, 20), #2
    nn.LeakyReLU(negative_slope=alpha) #3

)
nn.init.kaiming_uniform_(model[0].weight, alpha,
nonlinearity="leaky_relu")
nn.init.kaiming_uniform_(model[2].weight, alpha,nonlinearity="leaky_relu")

Parameter containing:
tensor([[ 4.9998e-02, -3.3787e-01,  2.4931e-01,  1.6044e-01,  1.0954e-01,
         -1.1900e-01, -2.6393e-01, -2.3814e-02, -1.1398e-01, -2.9974e-01,
         -2.2311e-01,  8.4044e-02,  1.3133e-01, -1.7062e-01, -7.9332e-02,
         -2.4819e-01,  1.8707e-01,  6.4221e-02,  3.5588e-01, -1.8069e-01,
         -1.4538e-01, -1.6178e-01, -5.1229e-02, -9.6449e-02,  1.0971e-01,
          1.2273e-01,  1.4576e-01, -2.8029e-01, -3.0786e-01, -2.4953e-03,
         -2.4039e-01,  3.9186e-02,  3.1125e-01,  1.8436e-01,  9.8426e-02,
          2.7781e-01, -3.0853e-01,  5.8744e-02, -3.6567e-01,  5.4230e-02],
        [-1.9848e-01, -1.3743e-01, -2.0809e-01,  2.2251e-01, -3.5647e-01,
          2.0120e-01, -2.8244e-01,  3.7521e-01,  3.0276e-01,  3.2167e-01,
         -2.0457e-01, -1.7897e-01,  3.2935e-02, -9.4278e-02, -2.4729e-02,
          2.1192e-01, -7.2509e-02, -3.0899e-01, -3.6729e-01,  3.2523e-01,
          4.6109e-02,  3.7053e-01,  2.5673e-01,  7.3270e-02,  1.5803e-02,
          8.873

### Batch Normalization

In [8]:
model = nn.Sequential(
    nn.Flatten(),
    nn.BatchNorm1d(1 * 28 * 28),
    nn.Linear(1 * 28 * 28,  300),
    nn.ReLU(),
    nn.BatchNorm1d(300),
    nn.Linear(300, 100),
    nn.ReLU(),
    nn.BatchNorm1d(100),
    nn.Linear(100, 10)
)

In [9]:
dict(model[1].named_parameters()).keys()

dict_keys(['weight', 'bias'])

In [10]:
dict(model[1].named_buffers()).keys()

dict_keys(['running_mean', 'running_var', 'num_batches_tracked'])

In [11]:
dict(model[1].named_children()).keys()

dict_keys([])

In [12]:
dict(model[1].named_modules()).keys()

dict_keys([''])

Note:
- if BN layers before the activation functions, we can also remove the bias term from the previous `nn.Linear` layers by setting the `bias` hyperparameter to `False`.

In [13]:
Model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(1 * 28 * 28, 300, bias=False),
    nn.BatchNorm1d(300),
    nn.ReLU(),
    nn.Linear(300, 100, bias=False),
    nn.BatchNorm1d(100),
    nn.ReLU(),
    nn.Linear(100, 10)
)

### Layer Normalization

In [14]:
inputs = torch.randn(32, 3, 100, 200)   # batch of random RGB images
layer_norm = nn.LayerNorm([100, 200])
result = layer_norm(inputs)
result

tensor([[[[ 1.5721, -0.3887,  0.7529,  ..., -0.7004, -2.1591, -1.6256],
          [ 1.2004,  0.8965,  1.0289,  ...,  1.7998, -0.2363, -0.0741],
          [-0.6671,  1.9502,  1.7339,  ...,  2.0054, -1.1634, -1.3759],
          ...,
          [ 1.1025,  0.0425,  0.0028,  ...,  0.0170, -0.7579,  0.5444],
          [ 0.8016, -0.5858,  1.0427,  ...,  1.9641, -1.4146, -0.7240],
          [ 0.3892,  0.7675, -0.1230,  ..., -0.5847, -0.7083, -0.4808]],

         [[-0.3886,  1.2498, -1.2231,  ...,  0.6728, -0.6909, -0.6072],
          [-1.0267, -1.8356,  0.8037,  ..., -0.4846,  0.3353,  0.2586],
          [ 0.7139, -0.7177,  0.1755,  ..., -0.7801,  2.6463, -1.5613],
          ...,
          [ 0.3852, -0.6375,  0.4169,  ..., -0.2669, -0.7811, -0.5624],
          [ 0.3749,  1.0245, -0.8857,  ...,  0.7339, -0.9092,  0.2227],
          [ 1.0404, -0.2668, -0.2765,  ...,  0.8848, -2.0543,  1.2872]],

         [[-0.5118, -0.9053, -1.0927,  ...,  0.7028, -0.5895, -0.4834],
          [ 1.4567,  1.0434,  

In [15]:
# method 2 to perform the same thing as above!
means = inputs.mean(dim=[2,3], keepdim=True)  #shape [32, 3, 1, 1]
vars_ = inputs.var(dim=[2,3], keepdim=True, unbiased=False) # shape: same
stds = torch.sqrt(vars_ + layer_norm.eps) # eps is a smoothing term (1e-5)
result = layer_norm.weight * (inputs - means) / stds + layer_norm.bias

In [16]:
layer_norm = nn.LayerNorm([3, 100, 200])
result = layer_norm(inputs)

In [17]:
result

tensor([[[[ 1.5703, -0.3874,  0.7524,  ..., -0.6987, -2.1550, -1.6224],
          [ 1.1992,  0.8957,  1.0280,  ...,  1.7976, -0.2353, -0.0734],
          [-0.6654,  1.9478,  1.7319,  ...,  2.0029, -1.1609, -1.3731],
          ...,
          [ 1.1014,  0.0431,  0.0034,  ...,  0.0176, -0.7561,  0.5442],
          [ 0.8010, -0.5842,  1.0417,  ...,  1.9616, -1.4117, -0.7222],
          [ 0.3893,  0.7669, -0.1221,  ..., -0.5831, -0.7066, -0.4794]],

         [[-0.3946,  1.2438, -1.2291,  ...,  0.6667, -0.6969, -0.6132],
          [-1.0327, -1.8416,  0.7976,  ..., -0.4906,  0.3292,  0.2526],
          [ 0.7078, -0.7237,  0.1694,  ..., -0.7861,  2.6401, -1.5672],
          ...,
          [ 0.3792, -0.6435,  0.4108,  ..., -0.2729, -0.7871, -0.5684],
          [ 0.3689,  1.0184, -0.8917,  ...,  0.7279, -0.9152,  0.2167],
          [ 1.0343, -0.2728, -0.2825,  ...,  0.8787, -2.0602,  1.2811]],

         [[-0.5072, -0.9013, -1.0890,  ...,  0.7093, -0.5850, -0.4788],
          [ 1.4644,  1.0505,  

## Gradient Clipping

See the line nn.utils.clip_grad_norm_(...) in the training function in the next section.

# Reusing Pretrained Layers

### Transfer Learning with PyTorch

- X_train_A = images of all items except for T-shirts/tops and pullovers (classes 0 and 2 )
- X_train_B = first 20 images of Tshirt and Pullovers

The validation set and the test set are also split this way, but without restricting the number of images.

We will train a model on set A (classification task with 8 classes), and try to reuse it to tackle set B (binary classification). We hope to transfer a little bit of knowledge from task A to task B, since classes in set A (trousers, dresses, coats, sandals, shirts, sneakers, bags, and ankle boots) are somewhat similar to classes in set B (T-shirts/tops and pullovers). However, since we are using Linear layers, only patterns that occur at the same location can be reused (in contrast, convolutional layers will transfer much better, since learned patterns can be detected anywhere on the image)


In [18]:
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import fetch_openml


fashion_mnist = fetch_openml(name="Fashion-MNIST", as_frame=False) #as_frames = False for getting the numpy arrays instead of pandas dataframe/series
X = torch.FloatTensor(fashion_mnist.data.reshape(-1, 1, 28, 28) / 255.)
y = torch.from_numpy(fashion_mnist.target.astype(int))
in_B = (y == 0) | (y == 2)  # Pullover or T-shirt/top
X_A, y_A = X[~in_B], y[~in_B]
y_A = torch.maximum(y_A - 2, torch.tensor(0))  # [1,3,4,5,6,7,8,9] => [0,..,7]
X_B, y_B = X[in_B], (y[in_B] == 2).to(dtype=torch.float32).view(-1, 1)

train_set_A = TensorDataset(X_A[:-7_000], y_A[:-7000])
valid_set_A = TensorDataset(X_A[-7_000:-5000], y_A[-7000:-5000])
test_set_A = TensorDataset(X_A[-5_000:], y_A[-5000:])
train_set_B = TensorDataset(X_B[:20], y_B[:20])
valid_set_B = TensorDataset(X_B[20:5000], y_B[20:5000])
test_set_B = TensorDataset(X_B[5_000:], y_B[5000:])

train_loader_A = DataLoader(train_set_A, batch_size=32, shuffle=True)
valid_loader_A = DataLoader(valid_set_A, batch_size=32)
test_loader_A = DataLoader(test_set_A, batch_size=32)
train_loader_B = DataLoader(train_set_B, batch_size=32, shuffle=True)
valid_loader_B = DataLoader(valid_set_B, batch_size=32)
test_loader_B = DataLoader(test_set_B, batch_size=32)

In [19]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

device


'cpu'

In [21]:
%pip install torchmetrics


import torchmetrics

def evaluate_tm(model, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)
    return metric.compute()

def train(model, optimizer, loss_fn, metric, train_loader, valid_loader,
          n_epochs):
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    for epoch in range(n_epochs):
        total_loss = 0.0
        metric.reset()
        model.train()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            # Uncomment to activate gradient clipping:
            #nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred, y_batch)
        history["train_losses"].append(total_loss / len(train_loader))
        history["train_metrics"].append(metric.compute().item())
        history["valid_metrics"].append(
            evaluate_tm(model, valid_loader, metric).item())
        print(f"Epoch {epoch + 1}/{n_epochs}, "
              f"train loss: {history['train_losses'][-1]:.4f}, "
              f"train metric: {history['train_metrics'][-1]:.4f}, "
              f"valid metric: {history['valid_metrics'][-1]:.4f}")
    return history



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 21.8 MB/s eta 0:00:00
